# Practice Lab: LangGraph Deep Dive (Lesson 11.3)

Module 11 · 8 exercises · Reducers + prebuilt + fan-out + Send API + HITL + time-travel + deploy

Exercises 1-4 run locally (pure Python simulations).


# Lesson 11.3: LangGraph Deep Dive

8 exercises: 2E + 3M + 3C

Exercises 1-4 run **locally** (pure Python simulations of LangGraph concepts). Ex 5-8 are design.


In [ ]:
import operator
import time
import json
from copy import deepcopy


---
## Exercise 1 (Easy): Reducer Explorer


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import operator

def overwrite(cur, new): return new
def add_red(cur, new): return operator.add(cur, new)
def add_messages(cur, new):
    ids = {m.get("id") for m in cur if isinstance(m,dict) and "id" in m}
    result = list(cur)
    for msg in new:
        if isinstance(msg,dict) and msg.get("id") in ids:
            result = [msg if m.get("id")==msg["id"] else m for m in result]
        else: result.append(msg)
    return result
def custom(cur, new):
    merged = {**cur}
    for k,v in new.items():
        merged[k] = max(merged[k],v) if k in merged and isinstance(v,(int,float)) else v
    return merged

print("Reducer Explorer:")
print("\n  1. Overwrite:", overwrite("node_a","node_b"), "(last wins)")
print("  2. operator.add:", add_red(add_red([1,2],[3,4]),[5,6]))
msgs = add_messages(
    [{"id":"m1","content":"Hello"},{"id":"m2","content":"World"}],
    [{"id":"m2","content":"World EDITED"},{"id":"m3","content":"New"}])
print("  3. add_messages:")
for m in msgs: print(f"     {m['id']}: {m['content']}", "UPDATED" if "EDITED" in m["content"] else "")
print("  4. Custom:", custom({"score":75,"model":"flash"},{"score":82,"attempts":3}))
print("\n  No reducer + parallel = InvalidUpdateError!")


</details>


---
## Exercise 2 (Easy): create_react_agent vs Manual


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("create_react_agent vs Manual:")
print("  One-liner: agent = create_react_agent(model, tools)  [2 lines]")
print("  Manual: State + agent_node + should_continue + StateGraph + edges  [~15 lines]")
print("")
print("  Internals: [agent] -> tools_condition -> 'tools':[ToolNode] -> [agent]")
print("                                        -> 'end':END")
print("")
cases = [("Prototyping","create_react_agent"),("Simple ReAct","create_react_agent"),
         ("Custom routing","Manual"),("Extra nodes","Manual"),("Subgraphs","Manual"),("Non-ReAct","Manual")]
print(f"  {'Use Case':<22} {'Approach'}")
for c,a in cases: print(f"  {c:<22} {a}")
print("\n  Rule: start with create_react_agent, switch to manual for customization")


</details>


---
## Exercise 3 (Medium): Fan-Out / Fan-In


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time, operator

def web(q): time.sleep(0.2); return {"src":"web","res":"14999","conf":0.85}
def docs(q): time.sleep(0.15); return {"src":"docs","res":"14999","conf":0.95}
def db(q): time.sleep(0.25); return {"src":"db","res":"14999","conf":0.99}

print("Fan-Out / Fan-In:")
start=time.time()
results=[web("g"),docs("g"),db("g")]
seq_ms=(time.time()-start)*1000
par_ms=max(200,150,250)
print(f"  Sequential: {seq_ms:.0f}ms | Parallel: {par_ms}ms | Speedup: {seq_ms/par_ms:.1f}x")

# Reducer
state = []
for r in results: state = operator.add(state, [r])
best = max(state, key=lambda r: r["conf"])
agree = len(set(r["res"] for r in state)) == 1
print(f"  Reducer: {len(state)} results, best={best['src']}({best['conf']}), agree={agree}")

print(f"\n  Atomic failure: exception in 1 branch -> ALL fail, state rolled back")
print(f"  LangGraph: 137x speedup. Reducer MANDATORY on shared keys")


</details>


---
## Exercise 4 (Medium): Send API Map-Reduce


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time

def analyze(doc):
    time.sleep(0.1)
    words = len(doc["text"].split())
    return {"id":doc["id"],"words":words,"summary":doc["text"][:25]+"..."}

print("Send API Map-Reduce:")
for n in [3, 5, 10]:
    docs = [{"id":f"d{i+1}","text":f"Document {i+1} about GenAI engineering course module {i+1} advanced topics"} for i in range(n)]
    start = time.time()
    results = [analyze(d) for d in docs]
    seq_ms = (time.time()-start)*1000
    par_ms = 100
    total_words = sum(r["words"] for r in results)
    print(f"  {n} docs: seq={seq_ms:.0f}ms, par={par_ms}ms, speedup={seq_ms/par_ms:.1f}x, words={total_words}")

print(f"\n  Send pattern: return [Send('analyze', {{'doc':d}}) for d in state['docs']]")
print(f"  Key: dynamic parallelism at RUNTIME, each Send gets independent state")


</details>


---
## Exercise 5 (Medium): HITL Approval Workflow
Design/architecture. See practice-lab-11_3.html.


In [ ]:
# Exercise 5: HITL Approval Workflow
pass


<details><summary>Solution</summary>


In [ ]:
print('HITL: [research] -> [approve] -> [execute]')
print('approve node: decision = interrupt({plan: state.plan})')
print('Resume: graph.invoke(Command(resume="approved"), config)')
print('Node RESTARTS from beginning on resume!')
print('Side effects before interrupt() must be IDEMPOTENT')
print('Never wrap interrupt() in try/except')


</details>


---
## Exercise 6 (Challenge): Time-Travel Debugger
Design/architecture. See practice-lab-11_3.html.


In [ ]:
# Exercise 6: Time-Travel Debugger
pass


<details><summary>Solution</summary>


In [ ]:
print('Time-Travel: checkpoints at every superstep')
print('Browse: graph.get_state_history(config)')
print('Fork: graph.update_state(step3.config, fix, as_node="search")')
print('Resume: graph.invoke(None, fork_config)')
print('Backends: InMemory(dev) | SQLite(local) | Postgres(prod) | Redis(TTL) | DynamoDB(serverless)')


</details>


---
## Exercise 7 (Challenge): Multilingual Router
Design/architecture. See practice-lab-11_3.html.


In [ ]:
# Exercise 7: Multilingual Router
pass


<details><summary>Solution</summary>


In [ ]:
print('Multilingual Router: detect_language -> conditional_edges')
print('Sarvam: ChatOpenAI(model="sarvam-105b", base_url="https://api.sarvam.ai/v1")')
print('Zero code changes! Same LangChain ChatOpenAI class')
print('Cost: Sarvam FREE (1000rs credits) vs GPT-4o $3/MTok = ~60% savings')
print('DPDP consent gate MANDATORY before India Stack calls (250Cr penalty)')


</details>


---
## Exercise 8 (Challenge): Production Deploy
Design/architecture. See practice-lab-11_3.html.


In [ ]:
# Exercise 8: Production Deploy
pass


<details><summary>Solution</summary>


In [ ]:
print('Deploy: langgraph dev -> langgraph up -> langgraph build -> docker/Cloud Run')
print('LangSmith: LANGSMITH_TRACING=true (zero code, auto-traces every node)')
print('RetryPolicy: max_attempts=3, backoff_factor=2, jitter=True')
print('Streaming: updates(delta) | messages(tokens) | custom(progress) | events(debug)')


</details>
